# From Grad-CAM heatmaps to saliency-guided augmentation

<a href="https://colab.research.google.com/github/bnnr-team/bnnr/blob/main/examples/integrations/gradcam_to_saliency_augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook picks up exactly where a typical [pytorch-grad-cam](https://github.com/jacobgil/pytorch-grad-cam) call ends. You already know how to turn a model plus a target layer into a heatmap. Here we reuse that same saliency signal for something different: two training-time augmentations from [BNNR](https://github.com/bnnr-team/bnnr) called **ICD** and **AICD**.

**What you will do**

1. Load an ImageNet-pretrained ResNet-18 and a few public-domain photos.
2. Compute a familiar Grad-CAM heatmap, the usual way.
3. Turn that same heatmap into ICD and AICD augmented images and compare them side by side.
4. See how the exact same call drops into a plain PyTorch training loop.

Everything runs in a couple of minutes on a free Colab CPU. No GPU needed.

> BNNR is MIT licensed and uses `grad-cam` as a core dependency. If you use the saliency path in your own work, please cite both projects. BibTeX is at the bottom.

## What are ICD and AICD?

Standard "coarse dropout" (also known as cutout) hides random rectangles of an image during training, so the model cannot lean on a single patch of pixels. It works, but it is blind: it erases background just as often as the object.

ICD and AICD make that idea saliency aware. They take a Grad-CAM map, split the image into a grid of small tiles, and choose which tiles to hide based on how important each tile is to the prediction.

* **ICD (Intelligent Coarse Dropout)** hides the *most* salient tiles. The region the model relies on is partly removed, so the network is pushed to use broader context and secondary cues instead of one shortcut. Think of it as a targeted, harder cutout.
* **AICD (Anti-ICD)** does the opposite. It hides the *least* salient tiles and keeps the important region intact. This perturbs background and context while protecting the object, which helps when you want the model to stay anchored on the right region or to become robust to background changes.

Instead of filling hidden tiles with hard black boxes, both augmentations support softer fills: a blurred copy of the original, local or global mean color, or noise. That keeps the image statistics closer to natural and avoids sharp artificial edges.

The knobs you will use most:

| Parameter | What it controls |
|-----------|------------------|
| `threshold_percentile` | How much of the saliency range counts as important. Higher means fewer tiles hidden by ICD. |
| `tile_size` | Grid granularity in pixels. Smaller tiles give finer masks. |
| `fill_strategy` | How hidden tiles are filled: `gaussian_blur`, `local_mean`, `global_mean`, `noise`, or `solid`. |

The saliency itself comes from `grad-cam`. BNNR computes it once per sample and caches it, so the augmentation stays cheap during training.

## Setup

In [ ]:
# On Colab this installs BNNR, which pulls in pytorch-grad-cam.
# In a local checkout where bnnr is already importable this is a no-op.
try:
    import bnnr  # noqa: F401
except ImportError:
    import subprocess
    import sys

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bnnr"], check=True)

In [ ]:
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

## The model and the pictures

We use a plain ResNet-18 with ImageNet weights and four CC0 (public domain) photos: a Labrador, a tabby cat, a cup of espresso, and a red sports car. Each maps to a clear ImageNet class, which keeps the saliency easy to read.

One small design choice makes the rest of the notebook clean. BNNR expects images in `[0, 1]` and asks you *not* to apply ImageNet normalization before ICD. So we wrap the network in a thin module that normalizes inside `forward`. Now the whole pipeline (predictions, Grad-CAM, ICD, AICD) speaks one language: plain `[0, 1]` images. The Grad-CAM target layer for ResNet is the last convolutional block, `layer4[-1]`.

In [ ]:
class NormalizedModel(nn.Module):
    """Accept images in [0, 1] and apply ImageNet normalization inside forward."""

    def __init__(self, backbone: nn.Module) -> None:
        super().__init__()
        self.backbone = backbone
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone((x - self.mean) / self.std)


weights = ResNet18_Weights.DEFAULT
backbone = resnet18(weights=weights).eval()
model = NormalizedModel(backbone).to(device).eval()
target_layers = [backbone.layer4[-1]]
categories = weights.meta["categories"]

preprocess = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),  # gives [0, 1], no ImageNet normalize here
    ]
)

In [ ]:
# Robust image loader: local repo copy first, then the bnnr repo on GitHub,
# then the original CC0 files on Wikimedia Commons as a last resort.
GITHUB_REF = "main"
RAW_BASE = f"https://raw.githubusercontent.com/bnnr-team/bnnr/{GITHUB_REF}/examples/integrations/assets"
WIKIMEDIA = {
    "dog_labrador.jpg": "https://commons.wikimedia.org/wiki/Special:FilePath/Portrait_of_a_labrador_retriever.jpg?width=768",
    "cat_tabby.jpg": "https://commons.wikimedia.org/wiki/Special:FilePath/Tabby_cat_with_blue_eyes-3336579.jpg?width=768",
    "espresso.jpg": "https://commons.wikimedia.org/wiki/Special:FilePath/Espresso_Coffee_01.jpg?width=768",
    "sports_car.jpg": "https://commons.wikimedia.org/wiki/Special:FilePath/Michael_Ligon_Red_Sports_Car_South_Beach_2022.png?width=768",
}
DEMO_FILES = ["dog_labrador.jpg", "cat_tabby.jpg", "espresso.jpg", "sports_car.jpg"]


def load_demo_image(name: str) -> Image.Image:
    for local in (Path("examples/integrations/assets") / name, Path("assets") / name):
        if local.exists():
            return Image.open(local).convert("RGB")
    Path("assets").mkdir(exist_ok=True)
    dst = Path("assets") / name
    errors = []
    for url in (f"{RAW_BASE}/{name}", WIKIMEDIA[name]):
        for attempt in range(3):
            try:
                req = urllib.request.Request(url, headers={"User-Agent": "bnnr-tutorial/1.0"})
                with urllib.request.urlopen(req, timeout=60) as resp:
                    dst.write_bytes(resp.read())
                return Image.open(dst).convert("RGB")
            except Exception as exc:  # noqa: BLE001
                errors.append(f"{url}: {exc}")
                time.sleep(2 * (attempt + 1))
    raise RuntimeError("Could not download {}:\n{}".format(name, "\n".join(errors)))


pil_images = [load_demo_image(n) for n in DEMO_FILES]
input_floats = torch.stack([preprocess(im) for im in pil_images]).to(device)  # B,3,224,224 in [0,1]

with torch.no_grad():
    probs = torch.softmax(model(input_floats), dim=1)
top_prob, top_idx = probs.max(dim=1)
labels = [int(i) for i in top_idx]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, im, idx, p in zip(axes, pil_images, labels, top_prob):
    ax.imshow(im.resize((224, 224)))
    ax.set_title(f"{categories[idx]}\n{float(p) * 100:.0f}% confident", fontsize=10)
    ax.axis("off")
fig.suptitle("Demo images and ResNet-18 top-1 predictions", fontsize=12)
fig.tight_layout()
plt.show()

## Step 1: the familiar Grad-CAM heatmap

Nothing new here. This is the standard `pytorch-grad-cam` call: pick a target class per image, run `GradCAM` on the target layer, and overlay the result. We target each image's own predicted class.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam_targets = [ClassifierOutputTarget(label) for label in labels]
with GradCAM(model=model, target_layers=target_layers) as cam:
    grayscale_cam = cam(input_tensor=input_floats, targets=cam_targets)  # B,H,W in [0,1]

rgb_floats = input_floats.permute(0, 2, 3, 1).cpu().numpy()  # B,H,W,3 in [0,1]
overlays = [show_cam_on_image(rgb_floats[i], grayscale_cam[i], use_rgb=True) for i in range(len(DEMO_FILES))]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for col in range(len(DEMO_FILES)):
    axes[0, col].imshow(rgb_floats[col])
    axes[0, col].set_title(categories[labels[col]], fontsize=10)
    axes[0, col].axis("off")
    axes[1, col].imshow(overlays[col])
    axes[1, col].axis("off")
axes[0, 0].set_ylabel("original")
axes[1, 0].set_ylabel("Grad-CAM")
fig.suptitle("Step 1: Grad-CAM overlays (pytorch-grad-cam)", fontsize=12)
fig.tight_layout()
plt.show()

## Step 2: the same saliency becomes ICD and AICD

Now we reuse that saliency. BNNR precomputes the Grad-CAM map once per sample into an `XAICache`, then ICD and AICD read from the cache to build their tile masks. The dataloader yields `(image, label, index)` so each cached map is keyed to the right sample.

We set `threshold_percentile=50` here so both ICD and AICD produce clearly visible masks. Higher values (like 70) make ICD more selective, which is useful in training, but AICD needs a lower threshold to have enough low-saliency tiles to act on (Grad-CAM maps are ReLU-ed, so many background tiles score exactly zero and only cross the threshold at lower percentiles).

Watch what happens:

* **ICD** blurs out the bright part of the heatmap (the dog's face, the cup, the car body). The subject is partly hidden.
* **AICD** blurs the dark part of the heatmap (the surroundings) and leaves the subject sharp.

In [ ]:
from torch.utils.data import DataLoader, Dataset

from bnnr.icd import AICD, ICD
from bnnr.xai_cache import XAICache


class IndexedTensors(Dataset):
    """Yield (image, label, index) so XAICache can key saliency per sample."""

    def __init__(self, tensors: torch.Tensor, labels: list[int]) -> None:
        self.tensors = tensors
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, i: int):
        return self.tensors[i], int(self.labels[i]), i


loader = DataLoader(IndexedTensors(input_floats.cpu(), labels), batch_size=len(DEMO_FILES))

cache = XAICache(Path("xai_cache_demo"))
cache.precompute_cache(
    model=model,
    train_loader=loader,
    target_layers=target_layers,
    n_samples=len(DEMO_FILES),
    method="gradcam",
    force_recompute=True,
    show_progress=False,
)

common = dict(
    model=model,
    target_layers=target_layers,
    cache=cache,
    explainer="gradcam",
    tile_size=16,
    threshold_percentile=50.0,
    fill_strategy="gaussian_blur",
    probability=1.0,
    random_state=SEED,
)
icd = ICD(**common)
aicd = AICD(**common)


def to_uint8_hwc(chw: torch.Tensor) -> np.ndarray:
    return (chw.permute(1, 2, 0).cpu().numpy() * 255).clip(0, 255).astype(np.uint8)


def to_float_rgb(arr: np.ndarray) -> np.ndarray:
    if arr.ndim == 2:
        arr = np.stack([arr] * 3, axis=-1)
    elif arr.shape[-1] == 1:
        arr = np.repeat(arr, 3, axis=2)
    return arr.astype(np.float32) / 255.0


icd_imgs, aicd_imgs = [], []
for i in range(len(DEMO_FILES)):
    base_u8 = to_uint8_hwc(input_floats[i])
    icd_imgs.append(to_float_rgb(icd.apply_with_label(base_u8, labels[i], sample_index=i)))
    aicd_imgs.append(to_float_rgb(aicd.apply_with_label(base_u8, labels[i], sample_index=i)))

In [ ]:
columns = ["original", "Grad-CAM", "ICD (hide salient)", "AICD (hide background)"]
fig, axes = plt.subplots(len(DEMO_FILES), 4, figsize=(13, 3.1 * len(DEMO_FILES)))
for row in range(len(DEMO_FILES)):
    axes[row, 0].imshow(rgb_floats[row])
    axes[row, 1].imshow(overlays[row])
    axes[row, 2].imshow(np.clip(icd_imgs[row], 0, 1))
    axes[row, 3].imshow(np.clip(aicd_imgs[row], 0, 1))
    axes[row, 0].set_ylabel(categories[labels[row]], fontsize=10)
    for col in range(4):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(columns[col], fontsize=11)
fig.suptitle("Step 2: one saliency map, two augmentations", fontsize=13)
fig.tight_layout()
plt.show()

### Fill strategies

The hidden tiles do not have to be black. Here is the same ICD mask on one image filled four different ways. `gaussian_blur` is a good default because it removes detail while keeping color and lighting close to the original.

In [ ]:
strategies = ["gaussian_blur", "local_mean", "noise", "solid"]
demo_idx = 0  # the Labrador
base_u8 = to_uint8_hwc(input_floats[demo_idx])

fig, axes = plt.subplots(1, len(strategies), figsize=(14, 3.6))
for ax, strategy in zip(axes, strategies):
    variant = ICD(
        model=model,
        target_layers=target_layers,
        cache=cache,
        explainer="gradcam",
        tile_size=16,
        threshold_percentile=50.0,
        fill_strategy=strategy,
        probability=1.0,
        random_state=SEED,
    )
    out = to_float_rgb(variant.apply_with_label(base_u8, labels[demo_idx], sample_index=demo_idx))
    ax.imshow(np.clip(out, 0, 1))
    ax.set_title(f"fill_strategy='{strategy}'", fontsize=10)
    ax.axis("off")
fig.suptitle("Same ICD mask, four fill strategies", fontsize=12)
fig.tight_layout()
plt.show()

## Step 3: dropping it into a training loop

This is the point of the whole exercise. Once the cache exists, the augmentation is one call on the batch: `apply_batch_with_labels(images, labels, sample_indices)`. Below is a minimal loop on the four demo images so you can see the shapes line up and the call fits where you would normally augment.

This is a mechanics demo, not a benchmark. Four images and a handful of steps say nothing about accuracy. For a real training loop see [`icd_plugin_minimal.py`](https://github.com/bnnr-team/bnnr/blob/main/examples/classification/icd_plugin_minimal.py), and for measured results see the [benchmark protocol](https://github.com/bnnr-team/bnnr/blob/main/docs/benchmarks.md).

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
label_tensor = torch.tensor(labels, device=device)
sample_indices = np.arange(len(DEMO_FILES))
batch_u8 = np.stack([to_uint8_hwc(input_floats[i]) for i in range(len(DEMO_FILES))])  # N,H,W,3

model.train()
for step in range(5):
    augmented = icd.apply_batch_with_labels(batch_u8, np.array(labels), sample_indices=sample_indices)
    batch = torch.from_numpy(augmented).permute(0, 3, 1, 2).float().div(255).to(device)

    optimizer.zero_grad()
    loss = criterion(model(batch), label_tensor)
    loss.backward()
    optimizer.step()
    print(f"step {step + 1}: loss={loss.item():.4f}")
model.eval()
print("Done. The ICD call slotted in exactly where a normal augmentation would.")

## When to reach for which

| Goal | Try |
|------|-----|
| Stop the model leaning on one patch, force it to use context | **ICD** |
| Keep the object intact, harden against background and context shifts | **AICD** |
| Finer or coarser masks | adjust `tile_size` |
| Hide more or fewer tiles | adjust `threshold_percentile` |
| Less artificial edges | keep `fill_strategy='gaussian_blur'` |

A common recipe is to use ICD with a moderate probability (so some batches stay clean) and to precompute the cache once at the start of training. Refresh the cache occasionally if the model changes a lot, since saliency drifts as the network learns.

## Where to go next

* [`docs/plugin_icd.md`](https://github.com/bnnr-team/bnnr/blob/main/docs/plugin_icd.md): the plug-in API in your own loop, including the no-cache warning and performance notes.
* [`examples/classification/icd_plugin_minimal.py`](https://github.com/bnnr-team/bnnr/blob/main/examples/classification/icd_plugin_minimal.py): a full MNIST training loop with ICD.
* [`examples/integrations/gradcam_to_icd_loop.py`](https://github.com/bnnr-team/bnnr/blob/main/examples/integrations/gradcam_to_icd_loop.py): the script version of Steps 1 and 2.
* [BNNR golden path](https://github.com/bnnr-team/bnnr/blob/main/docs/golden_path.md): the full trainer with automatic augmentation search.

## Citation

If you use the saliency-guided augmentations in research, please cite both BNNR and pytorch-grad-cam.

```bibtex
@software{walo2026bnnr,
  author  = {Walo, Mateusz and Morzhak, Diana and Zydorczyk, Dominika and Saczuk, Zuzanna},
  title   = {{BNNR}: Bulletproof Neural Network Recipe},
  year    = {2026},
  url     = {https://github.com/bnnr-team/bnnr},
  license = {MIT}
}

@misc{jacobgilpytorchcam,
  title        = {PyTorch library for CAM methods},
  author       = {Jacob Gildenblat and contributors},
  year         = {2021},
  publisher    = {GitHub},
  howpublished = {\url{https://github.com/jacobgil/pytorch-grad-cam}}
}
```

The demo images are CC0 (public domain) from Wikimedia Commons. Credits and source links are in [`assets/IMAGE_CREDITS.md`](https://github.com/bnnr-team/bnnr/blob/main/examples/integrations/assets/IMAGE_CREDITS.md).